<a href="https://colab.research.google.com/github/AWADKILLERB/jupyter/blob/master/Microscopic_Valley_Toolkit_Part6_QuantumMapping_ipynb_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Microscopic Valley Toolkit — Part 6: Mapping the Verified Hamiltonian onto a Quantum Circuit

**Title:** Microscopic Theory of Valley Splitting in Silicon
**Author:** Awad Mohamed
**Supervisor (Prospective):** Dr. Mark Friesen (University of Wisconsin–Madison)
**Project Status:** Stage VI — First Quantum-Computational Verification

---

## Purpose of this notebook

Every stage so far has been carried out on a conventional computer. This notebook
begins the originally stated goal of the project: encoding the literature-verified
atomic Hamiltonian of Part 3 as a quantum circuit, and confirming that a quantum
algorithm running on that circuit reproduces the same physics as the classical
diagonalization already trusted throughout this project.

A practical note on scope: IBM's Quantum Platform is not currently available,
under its own published access terms, to users located in Sudan. Everything in
this notebook is built and verified on a local, noise-free quantum simulator,
which executes the exact same mathematics a real quantum processor would.
Execution on actual quantum hardware remains a planned future step, pending
institutional access, and does not change anything about the validity of the
method demonstrated here.

## What this notebook establishes

Three things, each checked numerically before being trusted, in keeping with the
practice followed throughout this project:

1. That the ten-orbital Hamiltonian can be written as a proper fermionic
   operator and mapped onto ten qubits through the standard Jordan-Wigner
   transformation, without any loss of information.
2. That restricting the resulting qubit operator to the part of its Hilbert
   space corresponding to a single electron reproduces the original ten
   classical eigenvalues exactly.
3. That a quantum circuit respecting this single-electron restriction, combined
   with the variational quantum eigensolver, actually finds the correct lowest
   eigenvalue when run on the simulator.


In [ ]:
import numpy as np
import scipy.linalg as la
import warnings
warnings.filterwarnings("ignore")

!pip install qiskit-nature
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit.circuit.library import ExcitationPreserving
from qiskit.circuit import QuantumCircuit
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms.optimizers import COBYLA

np.set_printoptions(precision=6, suppress=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.6 MB/s eta 0:00:00


## Chapter 1 — The Verified Hamiltonian (unchanged from Part 3)

The same ten-orbital sp3s* Hamiltonian, built from the published Vogl,
Hjalmarson and Dow (1983) parameters for silicon and verified to machine
precision in Part 3, is reused here without any modification.


In [ ]:

params_sp3s = {
    "Es": -4.2000, "Ep": 1.7150, "Ess": 6.6850,
    "Vss_sigma": -8.3000, "Vsp_sigma": 5.7292, "Vssp_sigma": 5.3749,
    "a": 5.4310000000,
}
Vxx, Vxy = 1.7150, 4.5750
params_sp3s["Vpp_pi"] = Vxx - Vxy
params_sp3s["Vpp_sigma"] = Vxx + 2*Vxy
a = params_sp3s["a"]

def phase_factors(kx, ky, kz, a):
    kxp, kyp, kzp = kx*a/4.0, ky*a/4.0, kz*a/4.0
    g0 = np.cos(kxp)*np.cos(kyp)*np.cos(kzp) - np.sin(kxp)*np.sin(kyp)*np.sin(kzp)*1j
    g1 = -np.cos(kxp)*np.sin(kyp)*np.sin(kzp) + np.sin(kxp)*np.cos(kyp)*np.cos(kzp)*1j
    g2 = -np.sin(kxp)*np.cos(kyp)*np.sin(kzp) + np.cos(kxp)*np.sin(kyp)*np.cos(kzp)*1j
    g3 = -np.sin(kxp)*np.sin(kyp)*np.cos(kzp) + np.cos(kxp)*np.cos(kyp)*np.sin(kzp)*1j
    return g0, g1, g2, g3

def H_k_sp3s(kx, ky, kz, p):
    Es, Ep, Ess = p["Es"], p["Ep"], p["Ess"]
    Vss, Vsp, Vssp = p["Vss_sigma"], p["Vsp_sigma"], p["Vssp_sigma"]
    Vpp_s, Vpp_p = p["Vpp_sigma"], p["Vpp_pi"]
    a = p["a"]
    Vxx_ = (Vpp_s + 2*Vpp_p)/3.0
    Vxy_ = (Vpp_s - Vpp_p)/3.0
    g0, g1, g2, g3 = phase_factors(kx, ky, kz, a)
    HAA = np.diag([Es, Ep, Ep, Ep, Ess])
    HBB = np.diag([Es, Ep, Ep, Ep, Ess])
    HAB = np.array([
        [Vss*g0,   Vsp*g1,   Vsp*g2,   Vsp*g3,   0],
        [-Vsp*g1,  Vxx_*g0,  Vxy_*g3,  Vxy_*g2,  -Vssp*g1],
        [-Vsp*g2,  Vxy_*g3,  Vxx_*g0,  Vxy_*g1,  -Vssp*g2],
        [-Vsp*g3,  Vxy_*g2,  Vxy_*g1,  Vxx_*g0,  -Vssp*g3],
        [0,        Vssp*g1,  Vssp*g2,  Vssp*g3,  0]
    ], dtype=complex)
    return np.block([[HAA, HAB], [HAB.conj().T, HBB]])

H_test = H_k_sp3s(0.0, 0.0, 0.0, params_sp3s)
M = H_test.shape[0]
evals_classical = np.sort(la.eigvalsh(H_test))

print(f"Hamiltonian size: {H_test.shape[0]} orbitals")
print(f"Classical eigenvalues: {np.round(evals_classical, 6)}")


Hamiltonian size: 10 orbitals
Classical eigenvalues: [-12.5    -0.     -0.      0.      3.43    3.43    3.43    4.1     6.685
   6.685]


## Chapter 2 — From Orbitals to Qubits: the Jordan-Wigner Mapping

Every element of the tight-binding Hamiltonian is a single-particle hopping or
on-site term, so the whole Hamiltonian can be written exactly as a fermionic
operator built only from creation and annihilation operators acting on the ten
orbitals. This fermionic operator is then converted into an operator acting on
ten qubits using the standard, well-established Jordan-Wigner transformation, as
implemented in Qiskit Nature. No approximation is introduced at this step.


In [ ]:

fermionic_terms = {
    f"+_{p} -_{q}": H_test[p, q]
    for p in range(M) for q in range(M)
    if abs(H_test[p, q]) > 1e-12
}

fermionic_op = FermionicOp(fermionic_terms, num_spin_orbitals=M)
qubit_op = JordanWignerMapper().map(fermionic_op)

print(f"Number of qubits required: {qubit_op.num_qubits}")
print(f"Number of Pauli terms in the mapped operator: {len(qubit_op)}")


Number of qubits required: 10
Number of Pauli terms in the mapped operator: 19


## Chapter 3 — Verification: Do the Qubit and Classical Eigenvalues Match Exactly?

The full qubit operator acts on a Hilbert space of size two to the tenth power,
far larger than the original ten-dimensional orbital space, because it describes
every possible occupation pattern of the ten qubits, not only the physically
relevant case of a single electron. Restricting the operator to the subspace
spanned by the ten basis states with exactly one qubit excited recovers the
original single-particle problem exactly, and its eigenvalues should match the
classical ones to numerical precision.


In [ ]:

H_qubit_matrix = qubit_op.to_matrix()
print(f"Full qubit operator matrix size: {H_qubit_matrix.shape}")

single_excitation_indices = sorted(1 << i for i in range(M))
H_restricted = H_qubit_matrix[np.ix_(single_excitation_indices, single_excitation_indices)]
evals_quantum = np.sort(np.real(la.eigvalsh(H_restricted)))

print(f"\nEigenvalues from the qubit operator, restricted to one excitation:\n{np.round(evals_quantum, 6)}")
print(f"\nOriginal classical eigenvalues:\n{np.round(evals_classical, 6)}")

max_diff = np.max(np.abs(evals_quantum - evals_classical))
print(f"\nLargest difference between the two: {max_diff:.3e}")

assert max_diff < 1e-8
print("\nThe qubit mapping reproduces the classical spectrum to machine precision.")


Full qubit operator matrix size: (1024, 1024)

Eigenvalues from the qubit operator, restricted to one excitation:
[-12.5    -0.     -0.     -0.      3.43    3.43    3.43    4.1     6.685
   6.685]

Original classical eigenvalues:
[-12.5    -0.     -0.      0.      3.43    3.43    3.43    4.1     6.685
   6.685]

Largest difference between the two: 3.553e-15

The qubit mapping reproduces the classical spectrum to machine precision.


## Chapter 4 — A Particle-Conserving Quantum Circuit

Only states with exactly one qubit excited are physically meaningful, since the
tight-binding model describes a single electron. The circuit used here starts
from the reference state with qubit zero excited and the rest empty, and applies
a standard particle-conserving ansatz built from excitation-preserving two-qubit
gates, which by construction never leaves the single-excitation subspace no
matter how its parameters are varied.


In [ ]:

reference = QuantumCircuit(M)
reference.x(0)

ansatz = ExcitationPreserving(M, reps=2, insert_barriers=False)
full_circuit = reference.compose(ansatz)

print(f"Number of qubits: {full_circuit.num_qubits}")
print(f"Number of variational parameters: {ansatz.num_parameters}")
print(f"Circuit depth after decomposition: {full_circuit.decompose().depth()}")


Number of qubits: 10
Number of variational parameters: 120
Circuit depth after decomposition: 31


## Chapter 5 — Running the Variational Quantum Eigensolver on the Simulator

The circuit parameters are optimized to minimize the expectation value of the
qubit Hamiltonian, using a classical optimizer in the usual variational quantum
eigensolver loop. Everything here runs on a local statevector simulator, which
computes the same expectation values a real quantum processor would return in
the limit of no hardware noise.


In [ ]:

estimator = StatevectorEstimator()

def cost_function(params):
    bound = full_circuit.assign_parameters(params)
    job = estimator.run([(bound, qubit_op)])
    return job.result()[0].data.evs

np.random.seed(42)
x0 = np.random.uniform(-0.1, 0.1, ansatz.num_parameters)

optimizer = COBYLA(maxiter=2000, tol=1e-8)
result = optimizer.minimize(fun=cost_function, x0=x0)

ground_state_classical = evals_classical[0]

print(f"Ground-state energy found by the quantum algorithm: {result.fun:.6f} eV")
print(f"Correct classical ground-state energy:               {ground_state_classical:.6f} eV")
print(f"Absolute difference: {abs(result.fun - ground_state_classical):.3e} eV")

assert abs(result.fun - ground_state_classical) < 1e-3
print("\nThe quantum algorithm, run on the simulator, correctly reproduces the classical ground state.")


Ground-state energy found by the quantum algorithm: -12.499999 eV
Correct classical ground-state energy:               -12.500000 eV
Absolute difference: 6.478e-07 eV

The quantum algorithm, run on the simulator, correctly reproduces the classical ground state.


## Chapter 6 — Final Checkpoint

### What has been demonstrated

The literature-verified atomic Hamiltonian of silicon, exactly as validated in
Part 3, has been mapped onto a ten-qubit quantum circuit through a standard and
information-preserving transformation, and a variational quantum algorithm
running on that circuit has been shown to reproduce the correct ground-state
energy to a precision far tighter than any physical quantity of interest in this
project.

### What has not yet been demonstrated

This chapter locates only the lowest eigenvalue, corresponding to the deepest
valence state at the center of the Brillouin zone. The conduction-band state
relevant to valley splitting lies several eigenvalues higher in the spectrum,
and has not yet been extracted by the quantum algorithm. Execution has also been
limited to a noise-free simulator; real quantum hardware would introduce error
rates that have not been modeled here.

### Immediate next step

The natural continuation is to extend this demonstration using variational
quantum deflation, which repeats the same optimization while penalizing overlap
with previously found states, allowing the algorithm to climb the spectrum one
level at a time until it reaches the conduction-band state used throughout this
project, and eventually the valley states at plus and minus the self-derived
valley wavevector of Part 3.
